<a href="https://colab.research.google.com/github/iREMKARADA/AgeOfEmperors/blob/main/G%C3%B6rev_Men%C3%BCs%C3%BC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import os
import json
from datetime import datetime

# ─────────────────────────────────────────────
# STİLLER
# ─────────────────────────────────────────────
display(HTML("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Space+Mono:wght@400;700&family=DM+Sans:wght@300;400;600&display=swap');
:root {
    --bg:#0f0f14; --surface:#1a1a24; --border:#2e2e42;
    --accent:#7c6af7; --accent2:#f76aac;
    --text:#e8e8f0; --muted:#6b6b88;
    --success:#4ade80; --danger:#f87171;
}
.gorev-kart {
    background:var(--surface); border:1px solid var(--border);
    border-radius:12px; padding:12px 18px; margin:5px 0;
    display:flex; align-items:center; gap:14px;
    font-family:'DM Sans',sans-serif; color:var(--text);
    transition:border-color .2s;
}
.gorev-kart:hover { border-color:var(--accent); }
.gorev-kart.tamamlandi { opacity:0.5; border-color:#2e2e42; }
.gorev-kart.tamamlandi .gorev-metin { text-decoration:line-through; color:var(--muted); }
.gorev-numara { font-family:'Space Mono',monospace; color:var(--accent); font-size:12px; min-width:28px; font-weight:700; }
.gorev-metin  { flex:1; font-size:15px; }
.baslik       { font-family:'Space Mono',monospace; color:var(--text); font-size:22px; font-weight:700; margin:0 0 4px 0; }
.alt-baslik   { font-family:'DM Sans',sans-serif; color:var(--muted); font-size:13px; margin-bottom:20px; }
.bildirim     { font-family:'DM Sans',sans-serif; font-size:13px; padding:10px 16px; border-radius:8px; margin-top:10px; }
.bildirim.basari { background:#14291f; color:var(--success); border:1px solid #1e4a2e; }
.bildirim.hata   { background:#291414; color:var(--danger);  border:1px solid #4a1e1e; }
.bildirim.bilgi  { background:#1a1a24; color:var(--muted);   border:1px solid var(--border); }
.bildirim.geri   { background:#1a1a2e; color:#a78bfa;        border:1px solid #2e2e52; }
.sayac        { display:inline-block; background:var(--accent); color:#fff; font-family:'Space Mono',monospace; font-size:11px; border-radius:20px; padding:2px 10px; margin-left:8px; }
.bolme-baslik { font-family:'Space Mono',monospace; color:var(--muted); font-size:11px; text-transform:uppercase; letter-spacing:1.5px; margin:16px 0 8px 0; border-bottom:1px solid var(--border); padding-bottom:6px; }
.kucuk-not    { color:#6b6b88; font-family:'DM Sans',sans-serif; font-size:12px; margin:0 0 6px 0; }
.oncelik-yuksek { display:inline-block; background:#3b1a1a; color:#f87171; font-family:'Space Mono',monospace; font-size:10px; border-radius:4px; padding:2px 7px; margin-left:6px; }
.oncelik-orta   { display:inline-block; background:#2a2510; color:#fbbf24; font-family:'Space Mono',monospace; font-size:10px; border-radius:4px; padding:2px 7px; margin-left:6px; }
.oncelik-dusuk  { display:inline-block; background:#0f2318; color:#4ade80; font-family:'Space Mono',monospace; font-size:10px; border-radius:4px; padding:2px 7px; margin-left:6px; }
.tarih-badge    { display:inline-block; background:#1a1a2e; color:#a78bfa; font-family:'Space Mono',monospace; font-size:10px; border-radius:4px; padding:2px 7px; margin-left:6px; }
.tarih-gecmis   { background:#291414; color:#f87171; }
.durum-badge    { font-size:14px; }
.widget-button  { border-radius:8px !important; font-weight:bold !important; cursor:pointer !important; }
.ekle-btn   .widget-button { background:#7c6af7 !important; color:#fff !important; border:none !important; }
.sil-btn    .widget-button { background:#f76aac !important; color:#fff !important; border:none !important; }
.duz-btn    .widget-button { background:#38bdf8 !important; color:#1a1a24 !important; border:none !important; }
.yenile-btn .widget-button { background:#334155 !important; color:#e8e8f0 !important; border:none !important; }
.tamam-btn  .widget-button {
    background: transparent !important;
    color: #4ade80 !important;
    border: 2px solid #4ade80 !important;
    letter-spacing: 1px !important;
}
</style>
"""))

# ─────────────────────────────────────────────
# DOSYA ISLEMLERI
# Gorevler JSON formatinda /content/gorevler.json dosyasina kaydedilir.
# Her gorev: metin, tamamlandi, oncelik, tarih alanlarini icerir.
# ─────────────────────────────────────────────
DOSYA_ADI = "/content/gorevler.json"

def gorevleri_oku():
    """Dosyadan gorev listesini okur. Dosya yoksa bos liste doner."""
    try:
        if os.path.exists(DOSYA_ADI):
            with open(DOSYA_ADI, "r", encoding="utf-8") as f:
                return json.load(f)
    except Exception as e:
        print(f"Dosya okuma hatasi: {e}")
    return []

def gorevleri_kaydet(gorevler):
    """Gorev listesini JSON formatinda dosyaya yazar."""
    try:
        with open(DOSYA_ADI, "w", encoding="utf-8") as f:
            json.dump(gorevler, f, ensure_ascii=False, indent=2)
    except Exception as e:
        print(f"Dosya yazma hatasi: {e}")

# Uygulama baslarken mevcut gorevleri yukle
gorevler = gorevleri_oku()

# ─────────────────────────────────────────────
# HTML YARDIMCI FONKSIYONLAR
# ─────────────────────────────────────────────
ONCELIK_SINIF  = {"Yuksek": "oncelik-yuksek", "Orta": "oncelik-orta", "Dusuk": "oncelik-dusuk"}
ONCELIK_ETIKET = {"Yuksek": "YUKSEK", "Orta": "ORTA", "Dusuk": "DUSUK"}

def bildirim_html(mesaj, tur="basari"):
    """Renkli bildirim kutusu HTML'i uretir. tur: basari | hata | bilgi | geri"""
    return f'<div class="bildirim {tur}">{mesaj}</div>'

def tarih_badge_html(tarih_str):
    """
    Tarih string'inden renkli badge HTML'i uretir.
    Gecmis tarihler kirmizi, gelecek tarihler mor gosterilir.
    """
    if not tarih_str:
        return ""
    try:
        tarih = datetime.strptime(tarih_str, "%Y-%m-%d").date()
        gecmis = tarih < datetime.today().date()
        css = "tarih-badge tarih-gecmis" if gecmis else "tarih-badge"
        return f'<span class="{css}">Tarih: {tarih_str}</span>'
    except ValueError:
        return ""

def liste_html(gorevler, siralama="numara"):
    """
    Gorev listesini HTML olarak render eder.
    siralama: 'numara' | 'oncelik' | 'tarih' | 'durum'
    """
    if not gorevler:
        return bildirim_html("Henuz hic gorev eklenmemis.", "bilgi")

    oncelik_sira = {"Yuksek": 0, "Orta": 1, "Dusuk": 2}
    indeksli = list(enumerate(gorevler, 1))

    if siralama == "oncelik":
        indeksli = sorted(indeksli, key=lambda x: oncelik_sira.get(x[1].get("oncelik", "Orta"), 1))
    elif siralama == "tarih":
        indeksli = sorted(indeksli, key=lambda x: x[1].get("tarih", "") or "9999-99-99")
    elif siralama == "durum":
        indeksli = sorted(indeksli, key=lambda x: x[1].get("tamamlandi", False))

    html = ""
    for i, g in indeksli:
        tamamlandi   = g.get("tamamlandi", False)
        oncelik      = g.get("oncelik", "Orta")
        kart_cls     = "gorev-kart tamamlandi" if tamamlandi else "gorev-kart"
        durum_icon   = "[OK]" if tamamlandi else "[ ]"
        oncelik_cls  = ONCELIK_SINIF.get(oncelik, "oncelik-orta")
        oncelik_etiket = ONCELIK_ETIKET.get(oncelik, oncelik)
        oncelik_html = f'<span class="{oncelik_cls}">{oncelik_etiket}</span>'
        tarih_html   = tarih_badge_html(g.get("tarih", ""))
        html += (
            f'<div class="{kart_cls}">'
            f'<span class="gorev-numara">#{i:02d}</span>'
            f'<span class="durum-badge">{durum_icon}</span>'
            f'<span class="gorev-metin">{g.get("metin","")}</span>'
            f'{oncelik_html}{tarih_html}'
            f'</div>'
        )
    return html

# ─────────────────────────────────────────────
# DOGRULAMA FONKSIYONLARI
# ─────────────────────────────────────────────
def num_parse(text_widget):
    """
    Text widget'indan guvenli sekilde integer okur.
    Donus: int veya None (gecersiz / bos giris durumunda)
    """
    try:
        return int(text_widget.value.strip())
    except ValueError:
        return None

def numara_dogrula(deger):
    """
    Kullanicinin girdigi numarayi liste sinirlarinagore dogrular.
    Doner: (gecerli_mi: bool, hata_mesaji: str, indeks: int)
    indeks: 0 tabanli liste indeksi, hata durumunda -1
    """
    if not gorevler:
        return False, "Gorev listesi bos, islem yapilamaz.", -1
    if deger < 1 or deger > len(gorevler):
        return False, (
            f"#{deger} numarali gorev bulunamadi. "
            f"Gecerli aralik: 1 - {len(gorevler)}."
        ), -1
    return True, "", deger - 1

def tarih_dogrula(tarih_str):
    """
    YYYY-MM-DD formatini dogrular.
    Bos string gecerlidir (tarih girilmemis demektir).
    Doner: (gecerli_mi: bool, temizlenmis_deger: str)
    """
    tarih_str = tarih_str.strip()
    if not tarih_str:
        return True, ""
    try:
        datetime.strptime(tarih_str, "%Y-%m-%d")
        return True, tarih_str
    except ValueError:
        return False, ""

# ─────────────────────────────────────────────
# WIDGET TANIMLARI
# ─────────────────────────────────────────────
istyle  = {'description_width': '0px'}
ilayout = widgets.Layout(width='100%', height='40px')
blayout = widgets.Layout(width='100%', height='40px')

# --- Ekle bolumu ---
ekle_giris   = widgets.Text(placeholder='Yeni gorev yazin...', style=istyle, layout=ilayout)
ekle_oncelik = widgets.Dropdown(options=["Yuksek","Orta","Dusuk"], value="Orta", style=istyle, layout=ilayout)
ekle_tarih   = widgets.Text(placeholder='Son tarih: YYYY-MM-DD (opsiyonel)', style=istyle, layout=ilayout)
ekle_btn     = widgets.Button(description='+ Ekle', layout=blayout)
ekle_btn.add_class('ekle-btn')
ekle_cikti   = widgets.Output()

# --- Tamamla bolumu ---
# Text widget kullaniyoruz: BoundedIntText gecersiz degeri otomatik kisitlar,
# bu da hata mesajinin gosterilmesini engeller.
tamam_giris = widgets.Text(placeholder='Gorev numarasi yazin (ornek: 3)', style=istyle, layout=ilayout)
tamam_btn   = widgets.Button(description='Tamamla / Geri Al', layout=blayout)
tamam_btn.add_class('tamam-btn')
tamam_cikti = widgets.Output()

# --- Sil bolumu ---
# Text widget kullaniyoruz: ayni sebepten dolayi
sil_giris = widgets.Text(placeholder='Silinecek gorev numarasi (ornek: 3)', style=istyle, layout=ilayout)
sil_btn   = widgets.Button(description='Sil', layout=blayout)
sil_btn.add_class('sil-btn')
sil_cikti = widgets.Output()

# --- Duzenle bolumu ---
duz_giris_num = widgets.Text(placeholder='Duzenlenecek gorev numarasi (ornek: 2)', style=istyle, layout=ilayout)
duz_giris     = widgets.Text(placeholder='Yeni gorev metni...', style=istyle, layout=ilayout)
duz_oncelik   = widgets.Dropdown(options=["Yuksek","Orta","Dusuk"], value="Orta", style=istyle, layout=ilayout)
duz_tarih     = widgets.Text(placeholder='Yeni tarih: YYYY-MM-DD (bos = degismez)', style=istyle, layout=ilayout)
duz_btn       = widgets.Button(description='Guncelle', layout=blayout)
duz_btn.add_class('duz-btn')
duz_cikti     = widgets.Output()

# --- Liste ve Siralama ---
siralama_dd = widgets.Dropdown(
    options=[("Numara","numara"),("Oncelik","oncelik"),("Tarih","tarih"),("Durum","durum")],
    value="numara", style=istyle, layout=widgets.Layout(width='150px', height='36px')
)
yenile_btn = widgets.Button(description='Yenile',
                             layout=widgets.Layout(width='110px', height='36px'))
yenile_btn.add_class('yenile-btn')
liste_cikti = widgets.Output()

# ─────────────────────────────────────────────
# YARDIMCI FONKSIYONLAR
# ─────────────────────────────────────────────
def listeyi_goster():
    """Liste output widget'ini secili siralamaaya gore yeniden render eder."""
    with liste_cikti:
        clear_output(wait=True)
        tamamlanan  = sum(1 for g in gorevler if g.get("tamamlandi"))
        sayac       = f'<span class="sayac">{len(gorevler)}</span>'
        tamam_sayac = (
            f'<span class="sayac" style="background:#4ade80;color:#0f0f14">{tamamlanan} tamamlandi</span>'
            if tamamlanan else ""
        )
        display(HTML(
            f'<p class="bolme-baslik">Gorevler {sayac}{tamam_sayac}</p>'
            + liste_html(gorevler, siralama_dd.value)
        ))

# ─────────────────────────────────────────────
# OLAY ISLEYICILERI
# ─────────────────────────────────────────────
def on_ekle(b):
    """Yeni gorev ekler. Bos metin ve hatali tarih formati reddedilir."""
    metin     = ekle_giris.value.strip()
    tarih_str = ekle_tarih.value.strip()

    with ekle_cikti:
        clear_output(wait=True)

        # Bos metin kontrolu
        if not metin:
            display(HTML(bildirim_html("Gorev adi bos olamaz.", "hata")))
            return

        # Tarih format kontrolu
        tarih_ok, tarih_temiz = tarih_dogrula(tarih_str)
        if not tarih_ok:
            display(HTML(bildirim_html("Tarih formati hatali. Dogru format: YYYY-MM-DD (ornek: 2025-12-31)", "hata")))
            return

        gorev = {
            "metin"      : metin,
            "tamamlandi" : False,
            "oncelik"    : ekle_oncelik.value,
            "tarih"      : tarih_temiz
        }
        gorevler.append(gorev)
        gorevleri_kaydet(gorevler)
        ekle_giris.value = ''
        ekle_tarih.value = ''
        display(HTML(bildirim_html(f"<b>{metin}</b> eklendi. [{ekle_oncelik.value}]")))

    listeyi_goster()


def on_tamam(b):
    """
    Secili gorevi tamamlandi / tamamlanmadi olarak isaretler.
    Tamamlandiysa yesil (basari), geri alindiysa mor (geri) bildirim gosterir.
    """
    with tamam_cikti:
        clear_output(wait=True)

        # 1. Gecerli sayi kontrolu
        giren = num_parse(tamam_giris)
        if giren is None:
            display(HTML(bildirim_html("Lutfen gecerli bir sayi girin (ornek: 3).", "hata")))
            return

        # 2. Numara aralik kontrolu
        gecerli, hata, idx = numara_dogrula(giren)
        if not gecerli:
            display(HTML(bildirim_html(hata, "hata")))
            return

        # Durumu tersine cevir ve kaydet
        gorevler[idx]["tamamlandi"] = not gorevler[idx]["tamamlandi"]
        gorevleri_kaydet(gorevler)

        ad = gorevler[idx]['metin']
        if gorevler[idx]["tamamlandi"]:
            mesaj = f"<b>{ad}</b> tamamlandi olarak isaretlendi."
            tur   = "basari"
        else:
            mesaj = f"<b>{ad}</b> tamamlanmadi olarak geri alindi."
            tur   = "geri"

        tamam_giris.value = ''
        display(HTML(bildirim_html(mesaj, tur)))

    listeyi_goster()


def on_sil(b):
    """
    Secili gorevi siler.
    Text widget kullandigi icin kullanicinin tam yazdigi deger okunur,
    BoundedIntText'in otomatik kisitlamasi devre disi.
    """
    with sil_cikti:
        clear_output(wait=True)

        # 1. Gecerli sayi kontrolu
        giren = num_parse(sil_giris)
        if giren is None:
            display(HTML(bildirim_html("Lutfen gecerli bir sayi girin (ornek: 3).", "hata")))
            return

        # 2. Numara aralik kontrolu
        gecerli, hata, idx = numara_dogrula(giren)
        if not gecerli:
            display(HTML(bildirim_html(hata, "hata")))
            return

        silinen = gorevler.pop(idx)
        gorevleri_kaydet(gorevler)
        sil_giris.value = ''
        display(HTML(bildirim_html(f"<b>{silinen['metin']}</b> silindi.", "hata")))

    listeyi_goster()


def on_duzenle(b):
    """
    Secili gorevi gunceller.
    Tamamlanmis gorevler duzenlenemez, once geri alinmasi gerekir.
    Text widget kullandigi icin gecersiz numara hatasini dogru yakalar.
    """
    yeni_metin = duz_giris.value.strip()
    tarih_str  = duz_tarih.value.strip()

    with duz_cikti:
        clear_output(wait=True)

        # 1. Gecerli sayi kontrolu
        giren = num_parse(duz_giris_num)
        if giren is None:
            display(HTML(bildirim_html("Lutfen gecerli bir sayi girin (ornek: 2).", "hata")))
            return

        # 2. Numara aralik kontrolu
        gecerli, hata, idx = numara_dogrula(giren)
        if not gecerli:
            display(HTML(bildirim_html(hata, "hata")))
            return

        # 3. Tamamlanmis gorev kontrolu — duzenleme yapilamaz
        if gorevler[idx].get("tamamlandi", False):
            display(HTML(bildirim_html(
                f"<b>{gorevler[idx]['metin']}</b> zaten tamamlandi. "
                f"Duzenlemek icin once 'Tamamla / Geri Al' ile geri alin.", "hata"
            )))
            return

        # 4. Bos metin kontrolu
        if not yeni_metin:
            display(HTML(bildirim_html("Gorev metni bos olamaz.", "hata")))
            return

        # 5. Tarih format kontrolu
        tarih_ok, tarih_temiz = tarih_dogrula(tarih_str)
        if not tarih_ok:
            display(HTML(bildirim_html("Tarih formati hatali. Dogru format: YYYY-MM-DD (ornek: 2025-12-31)", "hata")))
            return

        # Tum kontroller gecildi — gorevi guncelle
        eski = gorevler[idx]["metin"]
        gorevler[idx]["metin"]   = yeni_metin
        gorevler[idx]["oncelik"] = duz_oncelik.value
        if tarih_temiz:
            gorevler[idx]["tarih"] = tarih_temiz

        gorevleri_kaydet(gorevler)
        duz_giris.value     = ''
        duz_tarih.value     = ''
        duz_giris_num.value = ''
        display(HTML(bildirim_html(f"<b>{eski}</b> degistirildi: <b>{yeni_metin}</b> [{duz_oncelik.value}]")))

    listeyi_goster()


# Buton ve dropdown baglantilari
ekle_btn.on_click(on_ekle)
tamam_btn.on_click(on_tamam)
sil_btn.on_click(on_sil)
duz_btn.on_click(on_duzenle)
yenile_btn.on_click(lambda b: listeyi_goster())
siralama_dd.observe(lambda change: listeyi_goster() if change['name'] == 'value' else None)

# ─────────────────────────────────────────────
# ARAYUZ DUZENI
# ─────────────────────────────────────────────
baslik = widgets.HTML("""
<div style="padding:24px 0 8px">
  <p class="baslik">Gorev Yoneticisi</p>
  <p class="alt-baslik">ipywidgets &nbsp;|&nbsp; /content/gorevler.json</p>
</div>
""")

sol = widgets.VBox([
    widgets.HBox([
        widgets.HTML('<p class="bolme-baslik" style="flex:1;margin:0">GOREV LISTESI</p>'),
        siralama_dd,
        yenile_btn
    ], layout=widgets.Layout(align_items='center', gap='8px')),
    liste_cikti
], layout=widgets.Layout(width='55%', padding='0 20px 0 0'))

sag = widgets.VBox([
    widgets.HTML('<p class="bolme-baslik">YENI GOREV EKLE</p>'),
    ekle_giris,
    widgets.HTML('<p class="kucuk-not">Oncelik</p>'),
    ekle_oncelik,
    widgets.HTML('<p class="kucuk-not">Son Tarih</p>'),
    ekle_tarih,
    ekle_btn, ekle_cikti,

    widgets.HTML('<p class="bolme-baslik" style="margin-top:20px">TAMAMLA / GERI AL</p>'),
    widgets.HTML('<p class="kucuk-not">Gorev numarasini yazin</p>'),
    tamam_giris, tamam_btn, tamam_cikti,

    widgets.HTML('<p class="bolme-baslik" style="margin-top:20px">GOREV SIL</p>'),
    widgets.HTML('<p class="kucuk-not">Silinecek gorev numarasini yazin</p>'),
    sil_giris, sil_btn, sil_cikti,

    widgets.HTML('<p class="bolme-baslik" style="margin-top:20px">GOREV DUZENLE</p>'),
    widgets.HTML('<p class="kucuk-not">Duzenlenecek gorev numarasini yazin</p>'),
    duz_giris_num,
    widgets.HTML('<p class="kucuk-not">Yeni metin</p>'),
    duz_giris,
    widgets.HTML('<p class="kucuk-not">Oncelik</p>'),
    duz_oncelik,
    widgets.HTML('<p class="kucuk-not">Son Tarih (bos birakilirsa degismez)</p>'),
    duz_tarih,
    duz_btn, duz_cikti,
], layout=widgets.Layout(width='45%', padding='0 0 0 20px', border_left='1px solid #2e2e42'))

display(baslik, widgets.HBox([sol, sag], layout=widgets.Layout(width='100%', padding='8px 0')))
listeyi_goster()

HTML(value='\n<div style="padding:24px 0 8px">\n  <p class="baslik">Gorev Yoneticisi</p>\n  <p class="alt-basl…